# పాఠం 09 - మెటాకాగ్నిషన్ డిజైన్ ప్యాటర్న్


## సెటప్

ఈ నోట్‌బుక్ Microsoft Agent ఫ్రేమ్‌వర్క్ ఉపయోగించి మెటాకాగ్నిషన్ డిజైన్_PATTERN ను చూపిస్తుంది.

**ముందస్తు అవసరాలు:**
- పర్యావరణ చిత్తరువుల ద్వారా కాన్ఫిగర్ చేసిన Azure OpenAI డిప్లాయ్‌మెంట్
- Azure CLI లాగిన్ అయి ఉంది (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## మెటాకాగ్నిషన్ అంటే ఏమిటి?

మెటాకాగ్నిషన్ అంటే **చింతించడం గురించి ఆలోచించడం**. AI ఏజెంట్ల సందర్భంలో, దీని అర్థం యెందంటే:

- తమ స్వంత అవుట్పుట్లు మరియు తర్క ప్రక్రియ పై **స్వయంస్పర్తింపు** చేయడం
- **పొరపాట్లు గుర్తించడం** మరియు నిశ్శబ్దంగా విఫలమవకుండా సాఫీగా మళ్ళీ నిలబడటం
- తమ స్పందనలు పూర్తిగా మరియు సహాయకరంగా ఉన్నాయా అని **మూల్యాంకనం** చేయడం
- మొదటి ప్రయత్నం పనిచేయకపోతే తమ వ్యూహాన్ని **అనుకూలపరచడం** (ఉదాహరణకు, బ్యాకప్ సిస్టమ్ కి అధికారం ఇవ్వడం)

మెటాకగ్నిటివ్ ఏజెంట్ కేవలం ప్రశ్నలకు జవాబు ఇవ్వదు — అది తన స్వంత పనితీరును పర్యవేక్షించి వెంటనే సర్దుబాటు చేస్తుంది.


## ప్రాధమిక మరియు బ్యాకప్ టూల్స్

ఒక సాధారణ మెటాకాగ్నిషన్ నమూనా **ఫాల్స్బ్యాక్ వ్యూహం**. ఏజెంట్ మొదట ప్రాధమిక టూల్ను ప్రయత్నిస్తుంది; అది విఫలమైతే (ఉదా: 404 లోపం), ఏజెంట్ విఫలమయ్యిందని గుర్తించి పారదర్శకంగా బ్యాకప్ టూల్‌కి మారుతుంది.

ఇది అసలు ప్రపంచ వ్యవస్థలను అనుకరిస్తుంది, అక్కడ ప్రాథమిక సేవలు అందుబాటులో లేకపోవచ్చు మరియు ఏజెంట్లు ప్రత్యామ్నాయ మార్గాన్ని ఎంచుకునే ముందు సమస్యను స్వయంగా గుర్తించాలి.

క్రింద రెండు ఫ్లైట్ లుకప్ టూల్స్‌ను నిర్వచిస్తాము:
- **ప్రాథమిక** — పారిస్, టోక్యో, మరియు బార్సిలోనాను కవర్ చేస్తుంది
- **బ్యాకప్** — బర్లిన్, సిడ్నీ, మరియు న్యూయార్క్ సిటీని కవర్ చేస్తుంది


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## తప్పు పునరుద్ధరణతో స్వీయ-ప్రతిబింబించే ఏజెంట్

క్రింద ఉన్న ఏజెంట్‌కు ప్రాథమిక ఫ్లైట్ సిస్టమ్‌ను ముందుగా ప్రయత్నించాలని, విఫలతలను గుర్తించి, స్పష్టంగా బ్యాకప్ సిస్టమ్‌కు మారమని సూచనలివ్వబడింది. ప్రతి ప్రతిస్పందన తర్వాత అది తాత్కాలికంగా స్వయంవిమర్శనం చేస్తూ యూజర్ ప్రశ్నకు పూర్తిగా సమాధానం ఇచ్చిందో లేదో తనిఖీ చేస్తుంది.


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## స్వీయ మౌల్యాంకన నమూనా

మేటాకాగ్నిషన్ యొక్క మరో పార్శ్వం **స్వీయ మౌల్యాంకనం**: పూర్తి సమాచారం, ఖచ్చితత్వం మరియు సహాయకత్వం కోసం ఒక ప్రత్యేక ఏజెంట్ (లేదా అదే ఏజెంట్ రెండవసారి) ఒక ప్రతిస్పందనను సమీక్షిస్తుంది.

క్రింద אנו `ResponseEvaluator` ఏజెంట్‌ని తయారుచేస్తాం, ఇది ట్రావెల్ ఏజెంట్ ప్రతిస్పందనలను మూడు పరిమాణాలలో గుణాంకం పెడుతుంది.


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## సారాంశం

ఈ పాఠంలో మీరు Microsoft Agent Framework ఉపయోగించి **మెటాకాగ్నిటివ్ ఏజెంట్లను** ఎలా నిర్మించాలో తెలుసుకున్నారు:

- **ఆత్మ-పరిశీలన**: వారి స్వీయ తర్కాన్ని పర్యవేక్షించే ఏజెంట్లు మరియు ఏం జరిగింది అనేదాన్ని పారదర్శకంగా కమ్యూనికేట్ చేస్తాయి.
- **తప్పిదీర్ఘ నివారణతో పొరపాటు పరిష్కారం**: ఏజెంట్ విఫలమయ్యే సరికి (ఉదా., 404 పొరపాట్లు) ప్రత్యామ్నాయ మూలాన్ని ఆటోమేటిక్ గా ప్రయత్నించే ప్రాథమిక + బ్యాకప్ టూల్ నమూనా.
- **ఆత్మ-మూల్యాంకనం**: ప్రతిస్పందనలను పూర్తిస్థాయిలో, ఖచ్చితత్వంలో మరియు సహాయకతలో మదింపుచేసే వేరొక మూల్యాంకకAgent.

ఈ నమూనాలు ఏజెంట్లను మరింత బలమైన, పారదర్శక, విశ్వసనీయంగా మారుస్తాయి — ఇది ఉత్పత్తి అమరికలకు ముఖ్యమైన గుణాలు.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
